# Census import data playground (HS4)

In [12]:

API_KEY = __import__("os").environ["CENSUS_API_KEY"]  # set in terminal, never typed here
BASE_URL = "https://api.census.gov/data/timeseries/intltrade/imports/hs"
COMM_LEVEL = "HS4"                    # HS4; can try HS6 later
COMM_LEVEL_PARAM = "COMM_LVL"         # query parameter name, verify on the Census page

COMMODITY_VAR, COUNTRY_VAR, DISTRICT_VAR = "I_COMMODITY", "CTY_CODE", "DISTRICT"

# Measure names: VERIFY with the variables cell below, then edit
MEASURES = {
    "charges":   "CNT_CHARGES_MO",   # containerized vessel charges (freight and insurance), USD
    "weight":    "CNT_WGT_MO",       # containerized vessel shipping weight, kg
    "cnt_value": "CNT_VAL_MO",       # containerized vessel import value, USD
    "gen_value": "GEN_VAL_MO",       # total general import value (customs value), USD
}

COUNTRIES = {"5700": "China"}        # add more later, for example "5520": "Vietnam", "5330": "India"
PULL_MONTHS = ["2023-09", "2023-10", "2023-11"]   # small to start, widen later
SEA_SHARE_MONTHS = ["2022-%02d" % m for m in range(1, 13)]  # 2022, for SeaShare

DEMO = API_KEY.startswith("PASTE")

In [13]:
import json, time, urllib.parse, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_json(url, tries=4):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                body = r.read().decode()
            return json.loads(body) if body.strip() else []
        except Exception as e:
            print("retry:", e); time.sleep(3 * (i + 1))
    return None

## 1. which variables exist

In [15]:

    v = get_json(BASE_URL + "/variables.json")["variables"]
    keys = ("CHARGE", "WGT", "WEIGHT", "CNT", "VES", "CIF", "GEN_VAL", "COMM", "CTY", "DIST", "LVL")
    for name, meta in sorted(v.items()):
        label = meta.get("label", "")
        if any(k in name.upper() or k in label.upper() for k in keys):
            print(f"{name:28s} {label}")

AIR_CHA_MO                   15-digit Air Charges
AIR_CHA_YR                   15-digit Year-to-date Air Charges
AIR_WGT_MO                   15-digit Air Shipping Weight
AIR_WGT_YR                   15-digit Year-to-Date Air Shipping Weight
CNT_CHA_MO                   15-digit Containerized Vessel Charges
CNT_CHA_YR                   15-digit Year-to-Date Containerized Vessel Charges
CNT_VAL_MO                   15-digit Containerized Vessel Value
CNT_VAL_YR                   15-digit Year-to-Date Containerized Vessel Value
CNT_WGT_MO                   15-digit Containerized Vessel Shipping Weight
CNT_WGT_YR                   15-digit Year-to-Date Containerized Vessel Shipping Weight
COMM_LVL                     4-character aggregation levels for commodity code.  HS2=2-digit HS totals.  HS4=4-digit HS totals. HS6=6-digit HS totals.  HS10=10-digit HS totals.
CON_CHA_MO                   15-digit Imports for Consumption, Charges
CON_CHA_YR                   15-digit Year-to-Date Import

In [17]:
MEASURES["charges"] = "CNT_CHA_MO"
print(MEASURES)

{'charges': 'CNT_CHA_MO', 'weight': 'CNT_WGT_MO', 'cnt_value': 'CNT_VAL_MO', 'gen_value': 'GEN_VAL_MO'}


In [18]:
import json, urllib.error, urllib.parse, urllib.request
import pandas as pd

def census_df(params):
    """Run one Census query and return a DataFrame, or None with the reason printed."""
    q = dict(params)
    q["key"] = API_KEY
    url = BASE_URL + "?" + urllib.parse.urlencode(q, safe=",*")
    try:
        with urllib.request.urlopen(url, timeout=120) as r:
            body = r.read().decode()
    except urllib.error.HTTPError as e:
        print("  Census error", e.code, ":", e.read().decode()[:300].replace("\n", " "))
        return None
    if not body.strip():
        print("  Empty response: no trade for this query")
        return None
    rows = json.loads(body)
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df = df.loc[:, ~df.columns.duplicated()]          # Census can repeat a column
    for col in MEASURES.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

base = {"time": "2023-10", "COMM_LVL": "HS4", "CTY_CODE": "5700"}
ids = "I_COMMODITY,CTY_CODE,DISTRICT,"
attempts = [
    ("all products, by district, detail rows",
     {**base, "get": ids + ",".join(MEASURES.values()), "I_COMMODITY": "*", "DISTRICT": "*", "SUMMARY_LVL": "DET"}),
    ("all products, by district",
     {**base, "get": ids + ",".join(MEASURES.values()), "I_COMMODITY": "*", "DISTRICT": "*"}),
    ("all products, country total (no district)",
     {**base, "get": "I_COMMODITY,CTY_CODE," + ",".join(MEASURES.values()), "I_COMMODITY": "*"}),
    ("one product (8517), total value only",
     {"time": "2023-10", "COMM_LVL": "HS4", "get": "I_COMMODITY,GEN_VAL_MO", "I_COMMODITY": "8517"}),
]

df = None
for label, params in attempts:
    print("Trying:", label)
    df = census_df(params)
    if df is not None:
        print("Worked:", label, "|", df.shape[0], "rows,", df.shape[1], "columns")
        break

if df is not None:
    display(df.head())
else:
    print("Nothing worked. Send me the error lines above.")

Trying: all products, by district, detail rows


KeyboardInterrupt: 

## 2. Test pull (one month, one country)

In [8]:
def pull(month, cty):
    fields = [COMMODITY_VAR, COUNTRY_VAR, DISTRICT_VAR] + list(MEASURES.values())
    q = {"get": ",".join(fields), "time": month, COMM_LEVEL_PARAM: COMM_LEVEL,
         COUNTRY_VAR: cty, COMMODITY_VAR: "*", DISTRICT_VAR: "*", "key": API_KEY}
    url = BASE_URL + "?" + urllib.parse.urlencode(q, safe=",*")
    rows = get_json(url)
    if not rows: return None
    return pd.DataFrame(rows[1:], columns=rows[0])

if not DEMO:
    test = pull("2023-10", "5700")
    print(None if test is None else test.head())

retry: HTTP Error 400: Bad Request
retry: HTTP Error 400: Bad Request
retry: HTTP Error 400: Bad Request
retry: HTTP Error 400: Bad Request


KeyboardInterrupt: 

## 3. Pull the months you chose (or build demo data)

In [ ]:
def make_demo(months):
    rng = np.random.default_rng(0)
    rows = []
    for m in months:
        for hs in ["8517", "8471", "9403", "6403", "8708"]:
            for d in ["10", "20", "30"]:
                w = rng.uniform(2e5, 2e6)
                shock = 1.0 + (0.8 if m >= "2023-11" and d == "10" else 0.0)
                ch = w * rng.uniform(0.8, 1.2) * shock
                val = w * rng.uniform(20, 80)
                rows.append({COMMODITY_VAR: hs, COUNTRY_VAR: "5700", DISTRICT_VAR: d, "time": m,
                             MEASURES["charges"]: ch, MEASURES["weight"]: w,
                             MEASURES["cnt_value"]: val * 0.9, MEASURES["gen_value"]: val})
    return pd.DataFrame(rows)

def pull_many(months):
    if DEMO:
        return make_demo(months)
    frames = []
    for m in months:
        for c in COUNTRIES:
            d = pull(m, c)
            if d is not None:
                d["time"] = m
                frames.append(d)
    return pd.concat(frames, ignore_index=True)

raw = pull_many(PULL_MONTHS)
raw.to_csv("census_pull_cache.csv", index=False)   # saved next to the notebook
print(raw.shape)
raw.head()

## 4. Calculations: freight cost per kg, value per kg, landed price

In [ ]:
def build_panel(df):
    df = df.rename(columns={COMMODITY_VAR: "hs4", COUNTRY_VAR: "origin", DISTRICT_VAR: "district", "time": "month"})
    for k, name in MEASURES.items():
        df[k] = pd.to_numeric(df[name], errors="coerce")
    df = df[(df["charges"] > 0) & (df["weight"] > 0) & (df["gen_value"] > 0)].copy()
    df["freight_per_kg"] = df["charges"] / df["weight"]
    df["value_per_kg"]   = df["gen_value"] / df["weight"]
    df["landed_per_kg"]  = (df["gen_value"] + df["charges"]) / df["weight"]
    df["freight_share_of_value"] = df["charges"] / df["gen_value"]   # for comparison only
    df["ln_fc"] = np.log(df["freight_per_kg"])
    return df

panel = build_panel(raw)
panel[["month", "origin", "district", "hs4", "freight_per_kg", "value_per_kg", "landed_per_kg", "ln_fc"]].head(10)

## 5. SeaShare by product (2022): containerized vessel value over total import value

In [ ]:
if DEMO:
    sea_raw = make_demo(["2022-06"])
else:
    sea_raw = pull_many(SEA_SHARE_MONTHS)

s = sea_raw.copy()
s["cnt_value"] = pd.to_numeric(s[MEASURES["cnt_value"]], errors="coerce")
s["gen_value"] = pd.to_numeric(s[MEASURES["gen_value"]], errors="coerce")
sea_share = s.groupby(COMMODITY_VAR)[["cnt_value", "gen_value"]].sum()
sea_share["sea_share"] = sea_share["cnt_value"] / sea_share["gen_value"].replace(0, np.nan)
sea_share[["sea_share"]].round(3)

## 6. Exposure placeholder

Exposure = SuezShare(origin, destination) x SeaShare(product). SuezShare is not built yet. The table below uses made up numbers so you can try the idea. Keys are customs districts, values are the share of the lane routed via Suez. Replace them with real shares later.

In [ ]:
suez_share_by_district = {"10": 0.6, "20": 0.2, "30": 0.0}   # PLACEHOLDER numbers, replace with real shares

panel = panel.merge(sea_share[["sea_share"]], left_on="hs4", right_index=True, how="left")
panel["suez_share"] = panel["district"].map(suez_share_by_district)
panel["exposure"] = panel["suez_share"] * panel["sea_share"]
panel[["month", "district", "hs4", "suez_share", "sea_share", "exposure"]].head()

## 7. Quick look: average ln freight cost by month, high versus low exposure

In [ ]:
panel["high_exposure"] = panel["exposure"] > panel["exposure"].median()
trend = panel.groupby(["month", "high_exposure"])["ln_fc"].mean().unstack()
ax = trend.plot(marker="o", figsize=(7, 4))
ax.set_ylabel("mean ln freight cost per kg"); ax.set_title("Raw look, not the event study")
plt.show()
trend

## 8. Next steps
* Widen `PULL_MONTHS` to Jan 2023 through Dec 2024 and add more origin countries.
* Replace the placeholder Suez shares with the real exposure measure.
* Then run the event study with the fixed effects from the brief: origin x destination x product, origin x month, product x month, destination x month.